In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# print(os.getcwd())
# print(os.path.abspath(base_path))


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [2]:
# ls

In [3]:
base_path = "../latency_correct_col_bi"

folders = {

    3: "mmlu_test_eval_service_3_300",
    5: "mmlu_test_eval_service_5_300",
    8: "mmlu_test_eval_service_8_300",
    10: "mmlu_test_eval_service_10_300",
    12: "mmlu_test_eval_service_12_300",
    14: "mmlu_test_eval_service_14_300"

}

In [4]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 5400 rows
K=5: loaded 7200 rows
K=8: loaded 9900 rows
K=10: loaded 11700 rows
K=12: loaded 13500 rows
K=14: loaded 15300 rows


In [5]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])
df_k12 = pd.DataFrame(data[12])
df_k14 = pd.DataFrame(data[14])


print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)
print("K=12:", df_k12.shape)
print("K=14:", df_k14.shape)

K=3: (5400, 32)
K=5: (7200, 34)
K=8: (9900, 37)
K=10: (11700, 39)
K=12: (13500, 41)
K=14: (15300, 43)


In [15]:
# df_k3.columns.tolist()# df_k3.columns

### combined df

In [6]:
df_combined = pd.concat([df_k3, df_k5, df_k8, df_k10, df_k12, df_k14], ignore_index=True)

print(df_combined.shape)
print(df_combined["total_service_count"].value_counts())
# df_combined.columns

(63000, 43)
14    15300
12    13500
10    11700
8      9900
5      7200
3      5400
Name: total_service_count, dtype: int64


## What does routing cost?

In [16]:
df_cost["energy_mwh"] = df_cost["energy_kwh"] * 1_000_000

g = df_cost.groupby(["routing_method", "model_name", "total_service_count"])["latency_ms"]

lat_all_k = g.agg(["median", "mean"]).unstack("total_service_count")
lat_all_k.round(2)

median                \
total_service_count                                       3      5      8    
routing_method model_name                                                    
bi-encoder     BAAI/bge-large-en-v1.5                  13.99  14.38  14.10   
               Qwen/Qwen3-Embedding-0.6B               28.09  28.29  28.60   
               Qwen/Qwen3-Embedding-4B                 36.41  36.41  36.50   
               Qwen/Qwen3-Embedding-8B                 36.33  36.21  36.25   
               sentence-transformers/all-MiniLM-L6-v2   4.75   4.83   4.91   
colbert        LiquidAI/LFM2-ColBERT-350M              14.25  14.42  14.39   
               colbert-ir/colbertv2.0                   9.22   9.27   9.34   
               lightonai/Reason-ModernColBERT          15.42  15.92  15.86   
               mixedbread-ai/mxbai-colbert-large-v1    15.74  15.99  15.86   

                                                                            \
total_service_count                                       10     12     14   
routing_method model_name                                                    
bi-encoder     BAAI/bge-large-en-v1.5                  14.17  14.12  14.17   
               Qwen/Qwen3-Embedding-0.6B               28.42  28.94  28.41   
               Qwen/Qwen3-Embedding-4B                 36.54  36.68  36.55   
               Qwen/Qwen3-Embedding-8B                 36.29  36.72  36.20   
               sentence-transformers/all-MiniLM-L6-v2   4.90   4.88   4.89   
colbert        LiquidAI/LFM2-ColBERT-350M              14.46  14.42  14.33   
               colbert-ir/colbertv2.0                   9.41   9.38   9.40   
               lightonai/Reason-ModernColBERT          15.86  15.83  16.01   
               mixedbread-ai/mxbai-colbert-large-v1    16.04  15.99  16.08   

                                                        mean                \
total_service_count                                       3      5      8    
routing_method model_name                                                    
bi-encoder     BAAI/bge-large-en-v1.5                  14.16  14.79  14.25   
               Qwen/Qwen3-Embedding-0.6B               30.82  28.70  28.82   
               Qwen/Qwen3-Embedding-4B                 36.74  36.56  36.69   
               Qwen/Qwen3-Embedding-8B                 36.52  36.36  36.75   
               sentence-transformers/all-MiniLM-L6-v2   5.95   4.87   5.00   
colbert        LiquidAI/LFM2-ColBERT-350M              15.00  14.68  14.55   
               colbert-ir/colbertv2.0                  10.69   9.23   9.32   
               lightonai/Reason-ModernColBERT          15.64  16.14  16.06   
               mixedbread-ai/mxbai-colbert-large-v1    17.37  16.24  16.05   

                                                                            
total_service_count                                       10     12     14  
routing_method model_name                                                   
bi-encoder     BAAI/bge-large-en-v1.5                  14.39  14.26  14.27  
               Qwen/Qwen3-Embedding-0.6B               28.59  29.17  28.47  
               Qwen/Qwen3-Embedding-4B                 36.67  36.98  36.63  
               Qwen/Qwen3-Embedding-8B                 36.53  36.99  36.32  
               sentence-transformers/all-MiniLM-L6-v2   5.00   4.91   4.90  
colbert        LiquidAI/LFM2-ColBERT-350M              14.78  14.63  14.53  
               colbert-ir/colbertv2.0                   9.49   9.34   9.36  
               lightonai/Reason-ModernColBERT          16.11  15.98  16.14  
               mixedbread-ai/mxbai-colbert-large-v1    16.28  16.08  16.18